K-means is an unsupervised machine learning technique that allow us to cluster data points. This enables us to find patterns in the data that can help us analyse it more effectively. K-means is an iterative algorithm, which means it improves the clusters step by step until they stop changing. Each run settles on a local best answer — which is why we run it several times and keep the best result.
To run a k-means clustering:
1.	Normalise the dataset
2.	Determine the optimum number of clusters using Elbow method
3.	Apply kmeans algorithm from Sci-Kit learn library
4.	Comprehend the clusters


<h1>Step 1: Understand the problem statement. </h1>
The dataset is quite simple with only two columns. It has students score and their effort in hours for that score in the test. We want to understand how the students are scattered around the two dimensions “Score” and “Effort”. Once the clusters are ready, we may want to dive in deeper with these group of students to understand their characteristics that help or hinder their scores. 


<h1>Step 2: Load, visualise and normalise the data</h1>
Load the dataset, look at the raw spread of the students, then rescale both columns to the 0–1 range with MinMax normalisation, so score (0–100) and
effort (hours) contribute equally to the distance calculation.

In [ ]:
# cap the maths library to one thread — avoids the Windows/MKL
# memory-leak warning in K-Means (must run before sklearn is imported)
import os
os.environ["OMP_NUM_THREADS"] = "1"

# import the libraries for data processing, plotting and clustering
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import minmax_scale

# read the dataset — 63 students, two columns: score and effortinhours
dataset = pd.read_csv("data/studentseffort.csv")
X = dataset.copy()

# visualise the raw data using a scatter plot
X.plot.scatter(x='score', y='effortinhours')

# MinMax normalisation: rescale each column to the 0-1 range so the two
# features contribute equally when K-Means measures distance
X_scaled = minmax_scale(X)


<h1>Step 3: Determine the optimum number of clusters using the Elbow Method </h1>
One of the most challenging parts to answer is how many optimum clusters are required of a dataset. In this example, it is quite simple because we only have two features : Scores and Effort. In the real-world scenarios, you can have more features and it may not be that clear on how the clusters are being formed. 
So, lets have a look into one of the most popular methods that determines the optimum number of clusters, that is the Elbow method.
The following code uses the Elbow method that will calculate the inertia or distances for 20 clusters and will graph the inertia graph to determine the best suitable k or number of clusters.


In [ ]:
# Elbow method to determine the optimum number of clusters
# fit K-Means for k = 1 to 19 and record the inertia for each k
inertia = []

for i in range(1, 20):                                          # i = 1, 2, ... 19
    kmeans = KMeans(n_clusters=i, n_init=10, random_state=42)
    kmeans.fit(X_scaled)                                        # cluster the NORMALISED data
    inertia.append(kmeans.inertia_)                             # total spread within the clusters

# plot k against inertia — the "elbow" is where the line stops dropping steeply
plt.figure(figsize=(12,6))
plt.plot(range(1, 20), inertia, marker='o')
plt.title('Elbow Curve')
plt.xlabel('Number of Clusters')
plt.ylabel('Squared Sum (Inertia)')


<h1>Step 4: Apply K-Means clustering </h1>
In this step we first determine the number of clusters. As you could see clearly from the above Elbow visualisation, it would be appropriate for us to choose 5 clusters. So now import KMeans from sklearn library to apply the KMeans algorithm to create clusters. The code below is given for your reference.


In [ ]:
# build the final model with the elbow's answer: 5 clusters
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)

# fit the normalised data and keep the cluster number assigned to each student
kmeans.fit(X_scaled)
labels = kmeans.labels_

# attach the cluster numbers to the (unscaled) data and plot,
# coloured by cluster — one colour per group of students
lbs = pd.DataFrame(labels)
df = pd.concat([X, lbs], axis=1)
df = df.rename(columns={0:'label'})

df.plot.scatter(x='score', y='effortinhours', c='label', colormap='Set1')


<h1>Step 5: Update data frame with the cluster information</h1>
Now, let’s say we want to append one more column in the dataset that will now have the cluster that each of the student belong to. So each row will now have a new value attached to, and that is the cluster. And we may also want to draw some histograms to visualise the distribution, we may write the following code:


In [ ]:
# add a cluster column to the original dataset so each student
# now carries the number of the group they belong to
dataset_cluster = pd.concat([dataset, pd.DataFrame({'cluster':labels})], axis=1)

# for each feature, draw one small histogram per cluster so we can
# compare the distribution of that feature across the five groups
for i in dataset.columns:
    plt.figure(figsize=(16,4))
    for j in range(5):
        plt.subplot(1, 5, j+1)
        cluster = dataset_cluster[dataset_cluster['cluster'] == j]
        cluster[i].hist(bins=20)
        plt.title('{}\nCluster {}'.format(i, j))
    plt.tight_layout()
plt.show()


<h1>What did we learn?</h1>

- **Normalising mattered.** Score runs 0–100 and effort 0–30; without MinMax scaling, score would dominate every distance and effort would barely count.
- **The elbow pointed to 5.** Inertia dropped steeply to k=5, then flattened — the data supports five groups, not more.
- **The five clusters are recognisable students.** High score on low effort (naturally strong), high score on high effort (hard work paying off), low score despite high effort (working hard but struggling — the group a teacher most wants to find), low score on low effort (disengaged), and the middle of the class — the biggest group.
- **Clustering finds the groups; we supply the meaning.** K-Means only saw dots and distances. Turning "cluster 2" into "hard workers who are struggling" — and deciding what to do about them — is the human part of the analysis.